In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [ ]:
df = pd.read_parquet(
    "s3://eagle-eye-ai-data/processed-data/candidate_risk_dataset_phase2/"
)

df.head()

In [ ]:
df.shape

In [ ]:
df["total_outstanding_debt"] = (
    df["total_outstanding_debt"]
    .fillna(0)
    .clip(lower=0)
)

df["total_outstanding_debt"] = np.log1p(
    df["total_outstanding_debt"]
)

df["total_outstanding_debt"].describe()

In [ ]:
X = df.drop(
    columns=[
        "target",
        "sk_id_curr"
    ]
)

y = df["target"]

print(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=y.value_counts()[0] / y.value_counts()[1],
    eval_metric="logloss"
)

model.fit(
    X_train,
    y_train
)

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=y.value_counts()[0] / y.value_counts()[1],
    eval_metric="logloss"
)

model.fit(
    X_train,
    y_train
)

In [ ]:
risk_scores = model.predict_proba(
    X_test
)[:, 1]

In [ ]:
results_phase2 = pd.DataFrame({
    "sk_id_curr": df.loc[X_test.index, "sk_id_curr"],
    "risk_score": risk_scores
})

results_phase2.head()

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(
    ascending=False
)

print(importance)

In [ ]:
results_phase2["risk_level"] = pd.cut(
    results_phase2["risk_score"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

In [ ]:
action_map = {
    "Low": "Continue Monitoring",
    "Medium": "Customer Relationship Manager Call",
    "High": "Risk Alert Email Sent"
}

results_phase2["action_taken"] = (
    results_phase2["risk_level"]
    .map(action_map)
)

In [ ]:
driver_features = df.loc[
    X_test.index,
    [
        "annual_income",
        "employment_years",
        "credit_history_length",

        "avg_payment_ratio",
        "payment_completion_rate",

        "total_shortfall",
        "underpayment_count",

        "max_credit_utilization",

        "previous_rejections",

        "missed_installments",

        "total_outstanding_debt"
    ]
].reset_index(drop=True)

results_phase2 = pd.concat(
    [
        results_phase2.reset_index(drop=True),
        driver_features
    ],
    axis=1
)

In [ ]:
def identify_risk_driver(row):

    if row["total_shortfall"] > 10000:
        return "Payment Shortfall"

    elif row["missed_installments"] > 5:
        return "Missed Installments"

    elif row["max_credit_utilization"] > 0.80:
        return "Credit Card Overuse"

    elif row["previous_rejections"] > 2:
        return "Multiple Previous Rejections"

    elif row["avg_payment_ratio"] < 0.90:
        return "Consistent Underpayment"

    elif row["payment_completion_rate"] < 0.80:
        return "Low Payment Completion Rate"

    elif row["total_outstanding_debt"] > 200000:
        return "High Outstanding Debt"

    elif row["employment_years"] < 1:
        return "Limited Employment History"

    elif row["credit_history_length"] < 2:
        return "Short Credit History"

    else:
        return "Multiple Moderate Risk Factors"


results_phase2["primary_risk_driver"] = (
    results_phase2.apply(
        identify_risk_driver,
        axis=1
    )
)

In [ ]:
results_phase2 = results_phase2[
    [
        "sk_id_curr",
        "risk_score",
        "risk_level",
        "primary_risk_driver",
        "action_taken"
    ]
]

In [ ]:
results_phase2.head(20)

In [ ]:
results_phase2["risk_level"].value_counts()

In [ ]:
top_risk_customers = (
    results_phase2
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(20)
)

top_risk_customers

In [ ]:
results_phase2["primary_risk_driver"].value_counts()

In [ ]:
results_phase2.to_csv(
    "phase2_risk_results.csv",
    index=False
)